In [ ]:
import sys
import os
import random
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import torchvision.transforms as transforms

#%matplotlib widget
import matplotlib.pyplot as plt

In [ ]:
sys.path.append(os.path.abspath("../../"))

from importlib import reload

import JunctionDetection.SkeletonizeDetect.segmentation_junction_detection
import Segmentation.Util.env_utils

reload(JunctionDetection.SkeletonizeDetect.segmentation_junction_detection)
reload(Segmentation.Util.env_utils)

In [ ]:
Segmentation.Util.env_utils.load_segmentation_env()

SEED = Segmentation.Util.env_utils.load_as("SEED", int, 42)

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")

HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
HIGHRES_MASK_DIR_NAME = os.getenv("HIGHRES_MASK_DIR_NAME", "masks_4096")

if RAW_DATA_DIR is None:
    raise ValueError("RAW_DATA_DIRenvironment variable not set.")

SAVE_PLOTS=True

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
def load_transform_image(path: Path, is_mask: bool = False):
    transform = transforms.Compose([transforms.ToTensor(),
                                    transforms.Lambda(lambda t: t.repeat(3, 1, 1) if t.shape[0] == 1 and not is_mask else t)])
    img = Image.open(path)
    return transform(img)


def show_mask(mask, ax, color: np.ndarray = None):
    if color is None:
        color = np.array([0.0, 1.0, 1.0, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def plot_img_mask_skeleton_junctions(image: torch.Tensor, mask: torch.Tensor, skeleton: np.ndarray, junction_coords: np.ndarray, figsize=(10, 10), save_path: Path = None):
    plt.figure(figsize=figsize)
    ax = plt.gca()

    ax.imshow(image.permute(1, 2, 0).cpu().numpy())
    show_mask(mask, ax)

    # ax.scatter(junction_coords[:, 0], junction_coords[:, 1], c='red', s=10)
    ax.plot(junction_coords[:, 0], junction_coords[:, 1], 'ro',
            markersize=20, markerfacecolor='none', markeredgewidth=2)

    # dilated_skeleton = dilation(skeleton, disk(2))
    overlay = np.zeros((*skeleton.shape, 4), dtype=np.uint8)
    overlay[skeleton > 0] = [255, 0, 0, 200]
    ax.imshow(overlay)

    ax.axis('off')

    if save_path is not None:
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, transparent=True)

    plt.show()
    plt.close()

In [ ]:
reload(JunctionDetection.SkeletonizeDetect.segmentation_junction_detection)

img_dir = Path(RAW_DATA_DIR) / HIGHRES_IMG_DIR_NAME
mask_dir = Path(RAW_DATA_DIR) / HIGHRES_MASK_DIR_NAME

image_paths = sorted(list(img_dir.glob("*.png")))
for img_path in image_paths:
    mask_path = mask_dir / img_path.name
    image = load_transform_image(img_path, is_mask=False)
    mask = load_transform_image(mask_path, is_mask=True)

    junction_coords, skeleton = JunctionDetection.SkeletonizeDetect.segmentation_junction_detection.detect_junctions_in_segmentation_mask(
        mask)

    print(f"Image: {img_path.name}, Detected junctions: {len(junction_coords)}")

    save_path = None
    if SAVE_PLOTS:
        save_path = Path(".") / f"{img_path.stem}_junctions_plt.png"

    plot_img_mask_skeleton_junctions(
        image, mask, skeleton, junction_coords, save_path=save_path)